# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

from agents.deals import ScrapedDeal
from agents.deals_common import DealSelection

import nest_asyncio

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

nest_asyncio.apply()

In [3]:
!py test_deals.py

🚀 Running in asynchronous mode (FETCH_ASYNC=True)...
✅ Retrieved 90 deals.
Title: Father's Day Gifts at Walmart for $25 or less + free shipping w/ $35
Details: We've pictured the No Boundaries Men's Soccer Sneakers for $15.98 ($4 savings). Choose pickup or spend $35 to avoid the $6.99 shipping charge. Buy Now at Walmart
Features: 
URL: https://www.dealnews.com/Fathers-Day-Gifts-at-Walmart-for-25-or-less-free-shipping-w-35/21742648.html?iref=rss-c142
Title: Anker Soundcore Life Q20 Hybrid Active Noise Cancelling Headphones for $40 + free shipping
Details: Apply stacking coupon codes "STARTSAVING" & "ANKER2025SALE" to drop the price. Buy Now at eBay
Features: active noise cancelling memory foam earpads Bluetooth 5.0 Model: A3025 UPC: 848061010022
URL: https://www.dealnews.com/products/Anker/Anker-Soundcore-Life-Q20-Hybrid-Active-Noise-Cancelling-Headphones/414984.html?iref=rss-c142
Title: Motorola G 5G 128GB Android Phone for Straight Talk for $40 + free shipping
Details: It's the same p


100%|██████████| 9/9 [00:00<?, ?it/s]


In [4]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<?, ?it/s]


In [5]:
len(deals)

90

In [6]:
deals[44].describe()

'Title: Vacmaster 6-Gallon 8A Ash Vacuum for $38 + free shipping\nDetails: No content-section found\nFeatures: \nURL: https://www.dealnews.com/products/Vacmaster/Vacmaster-6-Gallon-8-A-Ash-Vacuum/334517.html?iref=rss-c196'

In [7]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [8]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [9]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Father's Day Gifts at Walmart for $25 or less + free shipping w/ $35
Details: We've pictured the No Boundaries Men's Soccer Sneakers for $15.98 ($4 savings). Choose pickup or spend $35 to avoid the $6.99 shipping charge. Buy Now at Walmart
Features: 
URL: https://www.dealnews.com/Fathers-Day-Gifts-at-Walmart-for-25-or-less-free-shipping-w-35/21742648.html?iref=rss-c

In [10]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [11]:
result = get_recommendations()

In [12]:
print(type(result))
len(result.deals)

<class 'agents.deals_common.DealSelection'>


5

In [13]:
result.deals[1]

Deal(product_description='The ASUS Vivobook 14 is equipped with a 13th-generation Intel Core i7-1355U CPU, providing exceptional processing power for demanding tasks. Its 14-inch 1080p FHD display delivers vibrant visuals, perfect for media consumption and productivity. With 12GB of RAM and a 512GB SSD, this laptop ensures smooth multitasking and ample storage for your files. It runs on Windows 11 Home, making it ideal for both work and entertainment.', price=450.0, url='https://www.dealnews.com/products/ASUS/ASUS-Vivobook-14-13-th-Gen-i7-14-Laptop-w-12-GB-RAM-and-512-GB-SSD/489605.html?iref=rss-c39')

## ScannerAgent is using 'gpt-4o-mini'

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent(show_progress=True)
result = agent.scan()

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<?, ?it/s]


In [16]:
result.deals

[Deal(product_description='The Unlocked Google Pixel Fold is a cutting-edge 256GB Android smartphone that features a stunning foldable design, allowing for a larger display when opened. It runs on the latest Android operating system and is equipped with powerful hardware for smooth multitasking and gaming experiences. This device includes a unique camera system that offers exceptional photo quality, and it seamlessly integrates with Google services for enhanced productivity.', price=600.0, url='https://www.dealnews.com/products/Google/Unlocked-Google-Pixel-Fold-256-GB-Android-Smartphone/467718.html?iref=rss-c142'),
 Deal(product_description="The RingConn Gen 2 Smart Ring provides fitness tracking, heart rate monitoring, and sleep analysis all in a sleek and stylish design. This advanced wearable technology is balanced on comfort and functionality, making it ideal for everyday wear. It's compatible with various health apps, allowing you to sync data effortlessly and gain insights into y

In [17]:
print(result.deals[0])

product_description='The Unlocked Google Pixel Fold is a cutting-edge 256GB Android smartphone that features a stunning foldable design, allowing for a larger display when opened. It runs on the latest Android operating system and is equipped with powerful hardware for smooth multitasking and gaming experiences. This device includes a unique camera system that offers exceptional photo quality, and it seamlessly integrates with Google services for enhanced productivity.' price=600.0 url='https://www.dealnews.com/products/Google/Unlocked-Google-Pixel-Fold-256-GB-Android-Smartphone/467718.html?iref=rss-c142'


In [18]:
print(type(result))

<class 'agents.deals_common.DealSelection'>
